In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

DATA_PATH = "/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/ml_service/data/"
MODEL_PATH = "/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/ml_service/models/"

# Veriyi yukle
df = pd.read_csv(DATA_PATH + "normalize_veri.csv")

print("=" * 40)
print("VERI GENEL BILGI")
print("=" * 40)
print(f"Toplam satir: {len(df)}")
print(f"\nEtiket dagilimi:")
print(df["etiket"].value_counts())

VERI GENEL BILGI
Toplam satir: 21920

Etiket dagilimi:
etiket
uygun          17359
riskli          3885
uygun_degil      676
Name: count, dtype: int64


In [2]:
# Ozellikler ve hedef degiskeni belirle
ozellikler = ["max_sicaklik", "min_sicaklik", "yagis", "ruzgar_hizi"]
X = df[ozellikler]
y = df["etiket_kod"]

# Egitim ve test olarak böl (%80 egitim, %20 test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=" * 40)
print("VERI BOLME SONUCU")
print("=" * 40)
print(f"Egitim seti: {len(X_train)} satir")
print(f"Test seti  : {len(X_test)} satir")

# Random Forest modelini olustur ve egit
model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

model_rf.fit(X_train, y_train)
print("\nModel egitimi tamamlandi!")

VERI BOLME SONUCU
Egitim seti: 17536 satir
Test seti  : 4384 satir

Model egitimi tamamlandi!


In [3]:
# Test seti uzerinde tahmin yap
y_pred = model_rf.predict(X_test)

# LabelEncoder'i yukle
le = joblib.load(MODEL_PATH + "label_encoder.pkl")

print("=" * 40)
print("MODEL PERFORMANSI")
print("=" * 40)
print(classification_report(
    y_test, y_pred, 
    target_names=le.classes_
))

print("Karisiklik Matrisi:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=le.classes_,
    columns=le.classes_
)
print(cm_df)

MODEL PERFORMANSI
              precision    recall  f1-score   support

      riskli       1.00      1.00      1.00       777
       uygun       1.00      1.00      1.00      3472
 uygun_degil       1.00      1.00      1.00       135

    accuracy                           1.00      4384
   macro avg       1.00      1.00      1.00      4384
weighted avg       1.00      1.00      1.00      4384

Karisiklik Matrisi:
             riskli  uygun  uygun_degil
riskli          775      2            0
uygun             0   3472            0
uygun_degil       0      0          135


In [4]:
# Ozellik onemleri
ozellik_onemleri = pd.DataFrame({
    "ozellik": ozellikler,
    "onem": model_rf.feature_importances_
}).sort_values("onem", ascending=False)

print("=" * 40)
print("OZELLIK ONEMLERI")
print("=" * 40)
print(ozellik_onemleri)

# Risk skoru uret (0.0 - 1.0 arasi)
# predict_proba her sinif icin olasilik verir
y_proba = model_rf.predict_proba(X_test)
proba_df = pd.DataFrame(y_proba, columns=le.classes_)

# Risk skoru = riskli + uygun_degil olasiligi
proba_df["risk_skoru"] = proba_df["riskli"] + proba_df["uygun_degil"]

print("\n" + "=" * 40)
print("ORNEK RISK SKORLARI")
print("=" * 40)
print(proba_df.head(10).round(3))

OZELLIK ONEMLERI
        ozellik      onem
0  max_sicaklik  0.582096
1  min_sicaklik  0.358212
2         yagis  0.048552
3   ruzgar_hizi  0.011140

ORNEK RISK SKORLARI
   riskli  uygun  uygun_degil  risk_skoru
0    0.00   1.00          0.0        0.00
1    0.00   1.00          0.0        0.00
2    0.99   0.01          0.0        0.99
3    1.00   0.00          0.0        1.00
4    1.00   0.00          0.0        1.00
5    0.00   1.00          0.0        0.00
6    0.00   1.00          0.0        0.00
7    0.00   1.00          0.0        0.00
8    0.00   1.00          0.0        0.00
9    1.00   0.00          0.0        1.00


In [5]:
# Sahte veri ile test et
scaler = joblib.load(MODEL_PATH + "scaler.pkl")

test_verisi = pd.DataFrame([
    # Ideal bugday gunu
    {"max_sicaklik": 20.0, "min_sicaklik": 8.0,
     "yagis": 5.0, "ruzgar_hizi": 10.0},
    # Don riski olan gun
    {"max_sicaklik": 2.0, "min_sicaklik": -8.0,
     "yagis": 3.0, "ruzgar_hizi": 20.0},
    # Asiri sicak gun
    {"max_sicaklik": 40.0, "min_sicaklik": 28.0,
     "yagis": 0.0, "ruzgar_hizi": 5.0},
    # Yagisli ve soguk gun
    {"max_sicaklik": 5.0, "min_sicaklik": -3.0,
     "yagis": 30.0, "ruzgar_hizi": 35.0},
])

# Normalize et
test_normalize = pd.DataFrame(
    scaler.transform(test_verisi),
    columns=ozellikler
)

# Tahmin yap
tahminler = model_rf.predict(test_normalize)
skorlar = model_rf.predict_proba(test_normalize)
proba_test = pd.DataFrame(skorlar, columns=le.classes_)
proba_test["risk_skoru"] = proba_test["riskli"] + proba_test["uygun_degil"]

print("=" * 40)
print("MODEL TEST SONUCLARI")
print("=" * 40)
aciklamalar = [
    "Ideal bugday gunu",
    "Don riski olan gun",
    "Asiri sicak gun",
    "Yagisli ve soguk gun"
]
for i, (tahmin, aciklama) in enumerate(zip(tahminler, aciklamalar)):
    risk = proba_test["risk_skoru"].iloc[i]
    sinif = le.inverse_transform([tahmin])[0]
    print(f"\n{aciklama}:")
    print(f"  Sonuc    : {sinif}")
    print(f"  Risk Skoru: {risk:.2f}")

# Modeli kaydet
joblib.dump(model_rf, MODEL_PATH + "random_forest.pkl")
print("\n" + "=" * 40)
print("Model kaydedildi → models/random_forest.pkl")

MODEL TEST SONUCLARI

Ideal bugday gunu:
  Sonuc    : uygun
  Risk Skoru: 0.00

Don riski olan gun:
  Sonuc    : riskli
  Risk Skoru: 1.00

Asiri sicak gun:
  Sonuc    : uygun_degil
  Risk Skoru: 1.00

Yagisli ve soguk gun:
  Sonuc    : riskli
  Risk Skoru: 0.93

Model kaydedildi → models/random_forest.pkl
